# Benchmark COSMOS : rattrapage Qwen3-4B et Qwen3-8B

Ces deux modèles n'avaient pas fini lors du grand run. Ce notebook ne lance **qu'eux**, avec **les mêmes réglages que les 6 modèles déjà publiés** :
100 souvenirs de bruit (≈ 200 souvenirs au total), graines 0, 1, 2, 60 cas générés par graine, 3 souvenirs donnés au modèle en config COSMOS.

Il saute le test minuscule et le test de recherche sans IA (déjà faits).

**Avant de lancer** (panneau de droite) :
- **Session options → Accelerator : GPU T4 x2**
- **Session options → Internet : On**
- **Add Input** : un Dataset avec **`cosmos_bench_v2.zip`** (nouveau : il contient le correctif mémoire ci-dessous). Pas l'ancien zip.

Puis **Save Version → Save & Run All (Commit)**. Tu peux fermer l'onglet.

**À la fin**, onglet **Output** : `benchmark-qwen-rattrapage.zip` (résultats bruts + logs). Envoie-le-moi pour que je les ajoute au site.

In [ ]:
# 1) Diagnostic de la machine
!nvidia-smi -L
!python --version

In [ ]:
# 2) Récupère le code COSMOS depuis le Dataset
import glob, os, shutil, zipfile
dst = '/kaggle/working/cosmos'
shutil.rmtree(dst, ignore_errors=True)
hits = glob.glob('/kaggle/input/**/benchmark/run_all.py', recursive=True)
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), dst)
else:
    zips = sorted(glob.glob('/kaggle/input/**/cosmos_bench*.zip', recursive=True))[::-1]
    assert zips, 'Rien trouve dans /kaggle/input : clique sur Add Input et ajoute le Dataset'
    zipfile.ZipFile(zips[0]).extractall(dst)
%cd /kaggle/working/cosmos
!ls /kaggle/working/cosmos/benchmark

In [ ]:
# 3) Dépendances
!pip install -q fastembed
!pip install -q -U transformers accelerate
!python -c "import fastembed, transformers, torch; print('transformers', transformers.__version__, '| torch', torch.__version__, '| gpus', torch.cuda.device_count())"

## 4) Lancer les deux modèles

- **Qwen3-4B-Instruct-2507** : sur un GPU.
- **Qwen3-8B** : ensuite, réparti sur les deux GPU.

Pourquoi ils avaient planté : dès la 1re série, le prompt « .md complet » (≈ 5 000 tokens) saturait la mémoire du GPU.
Correctif : `COSMOS_PREFILL_CHUNK=1024` fait lire les longs prompts par morceaux de 1 024 tokens. Même calcul, mêmes réponses (testé), beaucoup moins de mémoire.
`expandable_segments` limite aussi la fragmentation de la mémoire GPU.
Les résultats sont sauvegardés après chaque graine : un modèle coupé laisse quand même des résultats partiels.

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['COSMOS_PREFILL_CHUNK'] = '1024'
!grep -q PREFILL_CHUNK benchmark/run.py && echo 'correctif present' || echo 'ATTENTION : ancien zip, ajoute cosmos_bench_v2.zip'
%cd /kaggle/working/cosmos
!python -m benchmark.run_all --models Qwen/Qwen3-4B-Instruct-2507,Qwen/Qwen3-8B --no-smoke --no-level1 --scales 100 --seeds 0,1,2 --synthetic 60 --budget-hours 9 --hard-limit-hours 11 --model-timeout-hours 5.5

## 5) Résultats

In [ ]:
# Résumé rapide de chaque modèle (taux de bonnes réponses par config)
import glob, json, os
base = '/kaggle/working/cosmos/benchmark/results'
for p in sorted(glob.glob(base + '/*.json')):
    if os.path.basename(p) in ('meta.json', 'run_status.json'):
        continue
    d = json.load(open(p, encoding='utf-8'))
    print(d.get('model'), '| partiel' if d.get('partial') else '| complet', '|', len(d.get('runs', [])), 'graine(s)')
    for scale, sm in (d.get('summary') or {}).items():
        for cfg, v in (sm.get('niveau2') or {}).items():
            acc, tok, sec = v['accuracy_repondables'][0], v['tokens_prompt_moy'][0], v['sec_moy'][0]
            print(f"   {cfg:8s} {acc*100:5.1f} % bonnes réponses | {tok:6.0f} tokens | {sec:5.2f} s")

In [ ]:
# Zip final : résultats bruts + logs (rapport en bonus, s'il se génère)
import zipfile
z = '/kaggle/working/benchmark-qwen-rattrapage.zip'
!cd /kaggle/working/cosmos && python -m benchmark.report --results {base} || echo "rapport non généré (pas grave : les résultats bruts suffisent)"
with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as out:
    for p in glob.glob(base + '/**/*', recursive=True):
        if os.path.isfile(p):
            out.write(p, os.path.relpath(p, base))
print(z, round(os.path.getsize(z) / 1e6, 2), 'Mo')
for n in zipfile.ZipFile(z).namelist():
    print(' ', n)